# Lighting

In [ ]:
import pandas as pd
raw_light = pd.read_csv('raw/street-lights-with-emitted-lux-level-council-owned-lights-only.csv')

In [14]:
light_df = raw_light[[col for col in raw_light.columns if raw_light[col].nunique() > 1]].copy()

light_df['latitude'] = light_df['geo_point_2d'].str.split(', ').str[0].astype(float)
light_df['longitude'] = light_df['geo_point_2d'].str.split(', ').str[1].astype(float)

# drop duplicates based on geo_shape, keeping only the record with latest ext_id
light_df = light_df.sort_values('ext_id', ascending=False)
light_df = light_df.drop_duplicates(subset=['geo_shape'])

light_df = light_df[['latitude', 'longitude', 'ext_id', 'label']].rename(columns={'label': 'emitted_lux_level'})
light_df.nunique()


latitude             97401
longitude            97398
ext_id               97405
emitted_lux_level     1016
dtype: int64

In [ ]:
# import time
# from geopy.geocoders import Nominatim
# from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

# geolocator = Nominatim(user_agent="my_geopy_app")

# def get_postcode(lat, lon):
#     retries = 3
#     for attempt in range(retries):
#         try:
#             location = geolocator.reverse((lat, lon), exactly_one=True)
#             address = location.raw.get('address', {})
#             suburb = (
#                 address.get('suburb') or
#                 address.get('town') or
#                 address.get('village') or
#                 address.get('hamlet')
#             )
#             postcode = address.get('postcode')
#             return pd.Series([suburb, postcode])
#         except (GeocoderTimedOut, GeocoderUnavailable):
#             time.sleep(2)
#         except Exception:
#             break
#     return pd.Series([None, None])

# results = []

# for idx, row in light_location_trim.iterrows():
#     lat, lon = row['lat_4'], row['long_4']
#     result = get_postcode(lat, lon)
#     results.append(result)
#     time.sleep(1)  # Respect OpenStreetMap rate limit (max 1 request/sec)

# light_location_trim[['suburb', 'postcode']] = results

# light_location_trim.to_csv("geocoded_light_locations.csv", index=False)

# Run for 302m 14.1s

In [47]:
# Geoended 
light_location = pd.read_csv('geocoded_light_locations.csv')
light_location.head()

,lat_4,long_4,suburb,postcode
0,-37.8191,144.9472,Docklands,3008.0
1,-37.8192,144.9472,Docklands,3008.0
2,-37.8192,144.9473,Docklands,3008.0
3,-37.8192,144.9474,Docklands,3008.0
4,-37.8193,144.9474,Docklands,3008.0


In [51]:
# data cleaning

light_location.loc[light_location['suburb'] == 'East Melbourne', 'postcode'] = 3002
light_location.loc[light_location['suburb'] == 'Parkville', 'postcode'] = 3052
light_location.loc[light_location['suburb'] == 'South Yarra', 'postcode'] = 3141

light_location['postcode'] = light_location['postcode'].astype('Int64')
light_location[['suburb', 'postcode']].value_counts()


suburb          postcode
Docklands       3008        1340
Parkville       3052         588
East Melbourne  3002         531
Melbourne       3000         495
Carlton         3053         380
Kensington      3031         194
Southbank       3006         143
Carlton North   3054         137
South Yarra     3141          82
Princes Hill    3054          14
Name: count, dtype: int64

In [52]:
light_df['lat_4'] = light_df['latitude'].round(4)
light_df['long_4'] = light_df['longitude'].round(4)

light_location_encoded = pd.merge(light_df, light_location, how='inner', on=['lat_4', 'long_4']) 
light_location_encoded.head()

,latitude,longitude,ext_id,emitted_lux_level,lat_4,long_4,suburb,postcode
0,-37.819103,144.947174,106037,18.280,-37.8191,144.9472,Docklands,3008
1,-37.819105,144.947177,106029,12.512,-37.8191,144.9472,Docklands,3008
2,-37.819108,144.947179,106028,12.414,-37.8191,144.9472,Docklands,3008
3,-37.819110,144.947182,106027,12.317,-37.8191,144.9472,Docklands,3008
4,-37.819112,144.947184,106026,12.414,-37.8191,144.9472,Docklands,3008


In [53]:
# 642 records records has no suburb/ postcode
light_location_encoded[light_location_encoded['suburb'].isnull()]

,latitude,longitude,ext_id,emitted_lux_level,lat_4,long_4,suburb,postcode
20399,-37.822551,144.951721,84499,45.552,-37.8226,144.9517,NaN,<NA>
20400,-37.822555,144.951721,84498,55.034,-37.8226,144.9517,NaN,<NA>
20401,-37.822558,144.951721,84497,60.020,-37.8226,144.9517,NaN,<NA>
20402,-37.822562,144.951721,84496,62.072,-37.8226,144.9517,NaN,<NA>
20403,-37.822566,144.951721,84495,61.779,-37.8226,144.9517,NaN,<NA>
...,...,...,...,...,...,...,...,...
21036,-37.820342,144.962673,83737,18.671,-37.8203,144.9627,NaN,<NA>
21037,-37.820344,144.962671,83736,18.671,-37.8203,144.9627,NaN,<NA>
21038,-37.820345,144.962669,83735,18.671,-37.8203,144.9627,NaN,<NA>
21039,-37.820347,144.962666,83734,18.573,-37.8203,144.9627,NaN,<NA>


In [28]:
# Geoencode based on full latitude and longitude

import time
from geopy.geocoders import Nominatim
from geopy.exc import GeocoderTimedOut, GeocoderUnavailable

geolocator = Nominatim(user_agent="my_geopy_app")

def get_postcode(lat, lon):
    retries = 3
    for attempt in range(retries):
        try:
            location = geolocator.reverse((lat, lon), exactly_one=True)
            address = location.raw.get('address', {})
            suburb = (
                address.get('suburb') or
                address.get('town') or
                address.get('village') or
                address.get('hamlet')
            )
            postcode = address.get('postcode')
            return pd.Series([suburb, postcode])
        except (GeocoderTimedOut, GeocoderUnavailable):
            time.sleep(2)
        except Exception:
            break
    return pd.Series([None, None])

light_location_encoded_2 = light_location_encoded[light_location_encoded['suburb'].isnull()].reset_index(drop=True)

results_2 = []

for idx, row in light_location_encoded_2.iterrows():
    lat, lon = row['latitude'], row['longitude']
    result = get_postcode(lat, lon)
    results_2.append(result)
    time.sleep(1)

light_location_encoded_2[['suburb', 'postcode']] = results_2

light_location_encoded_2.to_csv("geocoded_light_locations_2.csv", index=False)

In [54]:
light_location_encoded_2

,latitude,longitude,ext_id,emitted_lux_level,lat_4,long_4,suburb,postcode
0,-37.822551,144.951721,84499,45.552,-37.8226,144.9517,Docklands,3008
1,-37.822555,144.951721,84498,55.034,-37.8226,144.9517,Docklands,3008
2,-37.822558,144.951721,84497,60.020,-37.8226,144.9517,Docklands,3008
3,-37.822562,144.951721,84496,62.072,-37.8226,144.9517,Docklands,3008
4,-37.822566,144.951721,84495,61.779,-37.8226,144.9517,Docklands,3008
...,...,...,...,...,...,...,...,...
637,-37.820342,144.962673,83737,18.671,-37.8203,144.9627,Melbourne,3000
638,-37.820344,144.962671,83736,18.671,-37.8203,144.9627,Melbourne,3000
639,-37.820345,144.962669,83735,18.671,-37.8203,144.9627,Melbourne,3000
640,-37.820347,144.962666,83734,18.573,-37.8203,144.9627,Melbourne,3000


In [66]:
light_df_2 = light_location_encoded.merge(light_location_encoded_2[['latitude', 'longitude', 'suburb', 'postcode']], 
                                  how='left', 
                                  on=['latitude', 'longitude'])

light_df_2

# Fill missing suburb and postcode values in the original light_df with values from light_location_encoded_2
light_df_2['suburb'] = light_df_2['suburb_x'].fillna(light_df_2['suburb_y'])
light_df_2['postcode'] = light_df_2['postcode_x'].fillna(light_df_2['postcode_y'])

# Drop the redundant columns
light_df_2.drop(columns=['lat_4', 'long_4', 'suburb_x', 'postcode_x', 'suburb_y', 'postcode_y'], inplace=True)


In [67]:
light_df_2[['suburb', 'postcode']].value_counts().reset_index().sort_values(by='postcode')


,suburb,postcode,count
2,Melbourne,3000,14817
3,East Melbourne,3002,12765
6,Southbank,3006,5090
0,Docklands,3008,28029
5,Kensington,3031,5648
1,Parkville,3052,15479
4,Carlton,3053,9958
7,Carlton North,3054,3365
9,Princes Hill,3054,289
8,South Yarra,3141,1965


In [68]:
light_df_2.head()

,latitude,longitude,ext_id,emitted_lux_level,suburb,postcode
0,-37.819103,144.947174,106037,18.280,Docklands,3008
1,-37.819105,144.947177,106029,12.512,Docklands,3008
2,-37.819108,144.947179,106028,12.414,Docklands,3008
3,-37.819110,144.947182,106027,12.317,Docklands,3008
4,-37.819112,144.947184,106026,12.414,Docklands,3008


In [ ]:
light_df_2.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 97405 entries, 0 to 97404
Data columns (total 6 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   latitude           97405 non-null  float64
 1   longitude          97405 non-null  float64
 2   ext_id             97405 non-null  int64  
 3   emitted_lux_level  97405 non-null  float64
 4   suburb             97405 non-null  object 
 5   postcode           97405 non-null  Int64  
dtypes: Int64(1), float64(3), int64(1), object(1)
memory usage: 4.6+ MB


In [112]:
light_df_2.to_csv('street_light.csv',index=False)

# Pedestrian Data (Historical 2024-2025)

In [125]:
ped_i2 = pd.read_csv('iteration_2/pedestrian.csv')

In [152]:
ped_i3 = ped_i2.groupby(['location_id', 'period_of_time', 'sensor_description', 'sensor_name', 'latitude', 'longitude']).agg({
    'total_pedestrian_count': 'sum',
    'hours_covered': 'sum',
    'sensing_date': 'count'})\
    .rename(columns={'sensing_date': 'days_covered'})\
    .reset_index()

ped_i3.head()

,location_id,period_of_time,sensor_description,sensor_name,latitude,longitude,total_pedestrian_count,hours_covered,days_covered
0,1,1. Late Night (12am-6am),Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,85536,2124,355
1,1,2. Morning (6am-12pm),Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,959788,1784,355
2,1,3. Afternoon (12pm-6pm),Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,3077950,1239,350
3,1,4. Night (6pm-12am),Bourke Street Mall (North),Bou292_T,-37.813494,144.965153,900950,1083,354
4,2,1. Late Night (12am-6am),Bourke Street Mall (South),Bou283_T,-37.813807,144.965167,76304,2188,365


In [149]:
# Add geocoding on sensor_location
from geopy.geocoders import Nominatim

geolocator = Nominatim(user_agent="my_geopy_app")

def get_postcode(row):
    lat = row['latitude']
    lon = row['longitude']
    location = geolocator.reverse((lat, lon), exactly_one=True, zoom=14)
    full_address = location.raw.get('address', {})
    suburb = full_address['suburb']
    postcode = full_address['postcode']

    if postcode == '3000':
        suburb = 'Melbourne'
    return pd.Series([suburb, postcode])

sensor_location = ped_i3[['latitude', 'longitude']].drop_duplicates().reset_index(drop=True)
sensor_location[['suburb', 'postcode']] = sensor_location.apply(get_postcode, axis=1)

In [153]:
sensor_location.groupby(['suburb', 'postcode']).size().reset_index()

ped_i3 = pd.merge(ped_i3,
                  sensor_location[['latitude',
                                   'longitude',
                                   'suburb', 
                                   'postcode'
                                   ]], 
                  on=['latitude','longitude',], how='left')

ped_i3.groupby(['suburb', 'postcode']).size().reset_index()


,suburb,postcode,0
0,Carlton,3053,20
1,Docklands,3008,48
2,Kensington,3031,12
3,Melbourne,3000,236
4,North Melbourne,3051,12
5,Parkville,3052,12
6,South Wharf,3006,4
7,Southbank,3006,24
8,West Melbourne,3003,16


In [154]:
ped_i3.to_csv('pedestrian.csv', index=False)

# Police Station

In [1]:
import pandas as pd
raw_police = pd.read_csv('raw/Police_Stations.csv')

In [2]:
police_df = raw_police[raw_police['facility_state']=='VICTORIA']

police_df = police_df[['facility_name', 'abs_suburb', 'abs_postcode', 'facility_lat', 'facility_long', 'gnaf_formatted_address']]
police_df.rename(columns={'abs_suburb':'suburb',
                          'abs_postcode':'postcode',
                          'facility_lat':'lat',
                          'facility_long':'long',
                          'gnaf_formatted_address': 'formatted_address'
                          }, inplace=True)
police_df.head()

,facility_name,suburb,postcode,lat,long,formatted_address
0,BRANXHOLME POLICE STATION,BRANXHOLME,3302,-37.858419,141.798755,87-89 MONROE STREET
1,BRIAGOLONG POLICE STATION,BRIAGOLONG,3860,-37.843433,147.069628,19-21 AVON STREET
2,BRIDGEWATER POLICE STATION,BRIDGEWATER ON LODDON,3516,-36.601704,143.940009,38 PARK STREET
3,BRIGHT POLICE STATION,BRIGHT,3741,-36.731794,146.962049,7 PARK STREET
4,BROADFORD POLICE STATION,BROADFORD,3658,-37.203405,145.055423,156 HIGH STREET


In [3]:
police_df['suburb'] = police_df['suburb'].str.title()

In [ ]:
police_df[['suburb', 'postcode']].value_counts().reset_index()

,suburb,postcode,count
0,Williamstown,3016,2
1,Melbourne Airport,3045,2
2,Melbourne,3000,2
3,Abbotsford,3067,1
4,Numurkah,3636,1
...,...,...,...
326,Glen Waverley,3150,1
327,Gisborne,3437,1
328,Geelong,3220,1
329,Frankston North,3200,1


In [102]:
combined_df = pd.concat([ped_df[['suburb', 'postcode']].drop_duplicates(), light_df_2[['suburb', 'postcode']].drop_duplicates()])

combined_df['postcode'] = combined_df['postcode'].astype(int)
combined_df = combined_df.drop_duplicates().reset_index(drop=True)

combined_df.sort_values(by='postcode')

,suburb,postcode
0,Melbourne,3000
6,East Melbourne,3002
9,West Melbourne,3003
4,Melbourne,3004
2,South Wharf,3006
3,Southbank,3006
1,Docklands,3008
8,Kensington,3031
7,North Melbourne,3051
10,Parkville,3052


In [103]:
pd.merge(police_df, combined_df, on=['suburb', 'postcode'], how='inner')


,facility_name,suburb,postcode,lat,long,formatted_address
0,MELBOURNE WEST POLICE STATION,Docklands,3008,-37.813807,144.951254,313 SPENCER STREET
1,MELBOURNE NORTH POLICE STATION,North Melbourne,3051,-37.800308,144.954566,36-48 WRECKYN STREET
2,MELBOURNE EAST POLICE STATION,Melbourne,3000,-37.816554,144.966375,228-232 FLINDERS LANE
3,AFP - VIC - MELBOURNE OFFICE,Melbourne,3000,-37.811623,144.958143,383 LA TROBE STREET
4,SOUTHBANK POLICE STATION,Southbank,3006,-37.828319,144.960939,66 MORAY STREET


In [ ]:
melb_police_df = pd.merge(police_df, combined_df['postcode'], on=['postcode'], how='inner')

,facility_name,suburb,postcode,lat,long,formatted_address
0,FLEMINGTON POLICE STATION,Flemington,3031,-37.785119,144.932150,30 WELLINGTON STREET
1,MELBOURNE WEST POLICE STATION,Docklands,3008,-37.813807,144.951254,313 SPENCER STREET
2,MELBOURNE NORTH POLICE STATION,North Melbourne,3051,-37.800308,144.954566,36-48 WRECKYN STREET
3,MELBOURNE EAST POLICE STATION,Melbourne,3000,-37.816554,144.966375,228-232 FLINDERS LANE
4,AFP - VIC - MELBOURNE OFFICE,Melbourne,3000,-37.811623,144.958143,383 LA TROBE STREET


In [118]:
melb_police_df = melb_police_df.drop_duplicates()
melb_police_df

,facility_name,suburb,postcode,lat,long,formatted_address
0,FLEMINGTON POLICE STATION,Flemington,3031,-37.785119,144.932150,30 WELLINGTON STREET
1,MELBOURNE WEST POLICE STATION,Docklands,3008,-37.813807,144.951254,313 SPENCER STREET
2,MELBOURNE NORTH POLICE STATION,North Melbourne,3051,-37.800308,144.954566,36-48 WRECKYN STREET
3,MELBOURNE EAST POLICE STATION,Melbourne,3000,-37.816554,144.966375,228-232 FLINDERS LANE
4,AFP - VIC - MELBOURNE OFFICE,Melbourne,3000,-37.811623,144.958143,383 LA TROBE STREET
5,SOUTHBANK POLICE STATION,Southbank,3006,-37.828319,144.960939,66 MORAY STREET


In [119]:
melb_police_df.to_csv('melb_police_df.csv', index=False)

In [120]:
combined_df = pd.concat([combined_df[['suburb', 'postcode']].drop_duplicates(), melb_police_df[['suburb', 'postcode']].drop_duplicates()])
combined_df = combined_df.drop_duplicates().reset_index(drop=True)

combined_df.sort_values(by='postcode')

,suburb,postcode
0,Melbourne,3000
6,East Melbourne,3002
9,West Melbourne,3003
4,Melbourne,3004
2,South Wharf,3006
3,Southbank,3006
1,Docklands,3008
8,Kensington,3031
14,Flemington,3031
7,North Melbourne,3051


In [121]:
combined_df.to_csv('melb_postcode.csv', index=False)